# 623 SPP v20: independent global LSTM + direct learned actions

The input is the byte-identical v18 capture: chronological `DEMAND(addr)` and `CACHE_FILL(evicted_addr)` only. A single bounded global LSTM learns from the lossless 59-bit stream. Gate and count are deterministic, action ranks are independent, and deltas use a TRAIN-derived integer-exact vocabulary plus a rounded signed-log `OTHER` escape that gives broad bounded approximate coverage without endpoint guarantees. Fill alone uses a prior-corrected event/rank-keyed categorical draw. There is no normal-SPP template, page rule, candidate list, action feedback, probability threshold, or neural degree cap. The result remains a matched-input open-loop comparison.

In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select an NVIDIA A100 GPU runtime'
DEVICE_NAME=torch.cuda.get_device_name(0); assert 'A100' in DEVICE_NAME,f'Pinned run requires A100, observed {DEVICE_NAME}'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
torch.set_float32_matmul_precision('highest')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
REPO='/content/cache_arch'; TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n'); os.chmod(ASKPASS,0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
 if not os.path.isdir(REPO): subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
 else: subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally: pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(DEVICE_NAME,subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_spp/python/train_and_offline_infer.py'
MODEL_CONTRACT=json.loads(subprocess.check_output([sys.executable,SCRIPT,'--describe-model-points'],text=True))
TRAINING_CONFIG=MODEL_CONTRACT['training_config']
SOURCE_HASHES={'trainer_source_sha256':hashlib.sha256(pathlib.Path(SCRIPT).read_bytes()).hexdigest(),'model_contract_source_sha256':hashlib.sha256(pathlib.Path(f'{REPO}/formal_NN_training/experiments/623_offline_lstm_spp/python/model_contract.py').read_bytes()).hexdigest(),'threshold_free_policy_source_sha256':hashlib.sha256(pathlib.Path(f'{REPO}/formal_NN_training/common/threshold_free_policy.py').read_bytes()).hexdigest(),'decoder_sampler_source_sha256':hashlib.sha256(pathlib.Path(f'{REPO}/formal_NN_training/common/keyed_sampling.py').read_bytes()).hexdigest()}
RUN_ID=MODEL_CONTRACT['run_id']
DRIVE_ROOT=f'/content/drive/MyDrive/cache_prefetch_623_spp/{RUN_ID}'
INPUT_DIR=f'{DRIVE_ROOT}/colab_input'; OUTPUT_ROOT=f'{DRIVE_ROOT}/colab_output'; os.makedirs(DRIVE_ROOT,exist_ok=True)
name=f'{RUN_ID}.colab_input.tar.gz'; uploaded=files.upload(); assert sorted(uploaded)==[name],f'Select only {name}'
archive=f'{DRIVE_ROOT}/{name}'; pathlib.Path(archive).write_bytes(uploaded[name]); del uploaded
if os.path.isdir(INPUT_DIR): shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR,exist_ok=True)
with tarfile.open(archive,'r:gz') as handle:
 members=handle.getmembers()
 for member in members:
  parts=pathlib.PurePosixPath(member.name).parts
  assert not pathlib.PurePosixPath(member.name).is_absolute() and '..' not in parts and not member.issym() and not member.islnk() and (member.isfile() or member.isdir()),member.name
 handle.extractall(INPUT_DIR,members=members)
for record in pathlib.Path(f'{INPUT_DIR}/SHA256SUMS').read_text().splitlines():
 expected,item=record.split(maxsplit=1); item=item.lstrip('*')
 assert hashlib.sha256(pathlib.Path(f'{INPUT_DIR}/{item}').read_bytes()).hexdigest()==expected
print('verified byte-identical reused input',archive)

In [ ]:
TRACE=MODEL_CONTRACT['trace']; POLICY=MODEL_CONTRACT['policy']; ROLES=('train','guard','eval')
INPUTS={role:{'stream':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz','teacher':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_teacher_actions.csv.gz'} for role in ROLES}
for items in INPUTS.values():
 for path in items.values(): assert os.path.isfile(path),path
historical_manifest=json.loads(pathlib.Path(f'{INPUT_DIR}/collection_manifest.json').read_text())
expected_input={'status':'PASS','experiment_revision':MODEL_CONTRACT['experiment_revision'],'event_logger_schema':'623_causal_trigger_fill_v6','source_decision_effective_external_input':MODEL_CONTRACT['external_input_fields'],'model_input_is_causal_external_event_sequence_only':True,'cache_fill_feedback_used_as_raw_external_input':True,'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_request_rate_used_as_budget':False,'probability_threshold_used':False,'neural_degree_cap':None}
bad={k:(historical_manifest.get(k),v) for k,v in expected_input.items() if historical_manifest.get(k)!=v}; assert not bad,bad
assert historical_manifest['training_runtime_fields']==MODEL_CONTRACT['external_input_fields']==historical_manifest['inference_runtime_fields']
SOURCE=f'{INPUT_DIR}/spp_source_contract.json'
COLLECTION_MANIFEST_ROLE='historical_input_package_provenance_only'

In [ ]:
LOCAL_OUTPUT=f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_OUTPUT): shutil.rmtree(LOCAL_OUTPUT)
os.makedirs(LOCAL_OUTPUT)
SPECS=[{'tag':p['tag'],'family':p['family'],'size':p['size'],'pair':p['pair_id'],'maximum_parameters':p['maximum_parameter_count']} for p in MODEL_CONTRACT['points']]
assert [spec['size'] for spec in SPECS]==[16,32],SPECS
SWEEP=[]
for spec in SPECS:
 out=f"{LOCAL_OUTPUT}/{spec['tag']}"; cmd=[sys.executable,SCRIPT,'--policy',POLICY]
 for role in ROLES: cmd += [f'--{role}-stream',INPUTS[role]['stream'],f'--{role}-teacher-actions',INPUTS[role]['teacher']]
 cmd += ['--source-contract',SOURCE,'--out-dir',out,'--model-family',spec['family'],'--model-size',str(spec['size']),'--pair-id',spec['pair'],'--device','cuda','--seed',str(TRAINING_CONFIG['seed']),'--decoder-seed',str(TRAINING_CONFIG['decoder_seed']),'--epochs',str(TRAINING_CONFIG['epochs']),'--chunk-len',str(TRAINING_CONFIG['chunk_len']),'--accumulate-chunks',str(TRAINING_CONFIG['accumulate_chunks']),'--learning-rate',str(TRAINING_CONFIG['learning_rate'])]
 print('\nTraining',spec['tag'],' '.join(cmd),flush=True); subprocess.run(cmd,check=True)
 meta=json.loads(pathlib.Path(f'{out}/run_metadata.json').read_text())
 expected={'run_id':RUN_ID,'model_tag':spec['tag'],'model_family':'lstm','operation':MODEL_CONTRACT['operation'],'model_revision':MODEL_CONTRACT['model_revision'],'decoder_revision':MODEL_CONTRACT['decoder_revision'],'runtime_feature_count':59,'matched_normal_prefetcher':POLICY,'same_external_input_contract':True,'decoder_training_mode':'fully_supervised_independent_ranks_no_action_feedback','decoder_previous_teacher_action_used_as_input':False,'model_does_not_use_pc':True,'normal_policy_outputs_used_as_model_inputs':False,'probability_threshold_used':False,'neural_degree_cap':None,'same_page_rule_used_by_neural_inference':False,'delta_other_escape':MODEL_CONTRACT['delta_other_escape'],'delta_other_decode_precision':MODEL_CONTRACT['delta_other_decode_precision'],'full_signed_line_delta_range_reachable':False,'every_signed_line_delta_exactly_representable':False,'exact_delta_representability_scope':'train_vocabulary_only','request_count_sampling_performed':False,'fill_argmax_used':False,'fill_conditioned_on_actual_emitted_target':True,'global_chronological_lstm':True,'page_local_causal_state':False,'common_random_numbers_across_capacities':True,'guard_selected_checkpoint':True,'evaluation_used_for_selection':False,'evaluation_decode_count':1,'keyed_sampling_self_test':'PASS','integer_csv_exactness_self_test':'PASS','experiment_revision':MODEL_CONTRACT['experiment_revision']}
 expected.update(SOURCE_HASHES); expected.update(TRAINING_CONFIG)
 expected.update({'cublas_workspace_config':':4096:8','torch_deterministic_algorithms_enabled':True,'cudnn_deterministic':True,'cudnn_benchmark':False,'float32_matmul_precision':'highest','determinism_fail_closed':True,'cuda_device_name':DEVICE_NAME})
 bad={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}; assert not bad,bad
 assert meta['training_config']==TRAINING_CONFIG
 assert meta['model_point_contract']==MODEL_CONTRACT and 0<meta['parameter_count']<=spec['maximum_parameters']
 assert 0<meta['exact_delta_vocabulary_size']<=255 and meta['other_delta_class']==meta['exact_delta_vocabulary_size']
 assert meta['training_runtime_fields']==MODEL_CONTRACT['external_input_fields']==meta['inference_runtime_fields']
 hashes={meta.get('runtime_encoder_sha256'),meta.get('training_runtime_encoder_sha256'),meta.get('inference_runtime_encoder_sha256')}
 assert len(hashes)==1 and len(next(iter(hashes)))==64,hashes
 fills=meta['offline_nn_fill_level_counts']; assert sum(fills.values())==meta['offline_nn_entries']==meta['materialized_action_count']==meta['raw_predicted_action_count']
 assert meta['peak_persistent_recurrent_state_bytes']>0 and meta['dynamic_page_state_pages']==0
 SWEEP.append({k:meta[k] for k in ('model_tag','model_size','architecture_pair_id','parameter_count','parameter_storage_bytes_float32','peak_persistent_recurrent_state_bytes','selected_epoch','exact_delta_vocabulary_size','offline_normal_entries','offline_nn_entries','offline_nn_fill_level_counts','guard_selection_metrics','heldout_behavior_metrics')})
if os.path.isdir(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
pathlib.Path(f'{OUTPUT_ROOT}/sweep_manifest.json').write_text(json.dumps({'trace':TRACE,'input_revision':MODEL_CONTRACT['experiment_revision'],'model_revision':MODEL_CONTRACT['model_revision'],'collection_manifest_role':COLLECTION_MANIFEST_ROLE,'points':SWEEP},indent=2)+'\n')
print(json.dumps(SWEEP,indent=2))

In [ ]:
OUTPUT_ARCHIVE=f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(OUTPUT_ARCHIVE,'w:gz') as archive:
 for item in pathlib.Path(OUTPUT_ROOT).iterdir(): archive.add(item,arcname=item.name)
print('DONE',OUTPUT_ARCHIVE,os.path.getsize(OUTPUT_ARCHIVE),'bytes')
files.download(OUTPUT_ARCHIVE)

Copy the v20 output archive to the matching Sacramento run and launch replay. The input remains the byte-identical v18 capture; teacher actions are labels/comparator data only, and the comparison remains open-loop.